# 사내 문서 QA — v1: 검색 결과 확인

가상 사내 문서 PDF를 FAISS에 저장하고, 질문마다 유사도 검색으로 찾은 문서 3개를 출력한다. (Reranker 없음)

In [ ]:
CHUNK_SIZE=500
CHUNK_OVERLAP=100

In [ ]:
query_list = [
    "연차를 사용하려면 최소 며칠 전에 신청해야 하나요?",
    "서울 출장 시 숙박비는 최대 얼마까지 지원되나요?",
    "출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?",
    "회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?",
    "재택근무를 신청하려면 어떤 조건을 충족해야 하나요?",
    "직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?",
    "출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?",
    "개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?",
    "업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?",
    "제주도로 2박 3일 출장을 가는 경우 숙박비 지원 한도와 출장비 정산 기한을 각각 알려주세요.",
]

In [ ]:
from dotenv import load_dotenv 
import os

load_dotenv()
os.environ["LANGSMITH_PROJECT"] = "RAG_ONE_01"

In [ ]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i + 1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [ ]:
from langchain_community.document_loaders import PyPDFLoader 
from langchain_community.vectorstores import FAISS 
from langchain_openai import OpenAIEmbeddings 
from langchain_text_splitters import CharacterTextSplitter

# PDF 로드 
loader = PyPDFLoader("data/가상Tech_업무가이드.pdf")
documents = loader.load() 

# 문서 분할 
text_splitter = CharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

split_docs = text_splitter.split_documents(documents) 

# 임베딩 
embeddings = OpenAIEmbeddings()

# FAISS 벡터스토어 생성 
db = FAISS.from_documents(split_docs, embeddings)

In [ ]:
# 리트리버 생성
# Reranker 없이 FAISS 유사도 검색 결과를 그대로 사용한다. (Reranker 버전과 같은 문서 수로 비교하려고 k=3)
retriever = db.as_retriever(search_kwargs={"k": 3})

In [ ]:
for i, query in enumerate(query_list):
    docs = retriever.invoke(query)
    print(f"\n질문{i+1}. {query}\n")
    pretty_print_docs(docs)